# Drone Vision Pipeline – Notebook
**VisDrone Dataset: Human & Car Detection**

This notebook walks through:
1. Dataset understanding & visualisation
2. Model training (YOLOv8)
3. Detection + human counting
4. Object tracking
5. Evaluation


In [ ]:
import os, sys
sys.path.insert(0, '..')  # add project root

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Project modules
from src.dataset import (
    analyze_dataset, plot_dataset_stats,
    visualize_sample, convert_to_yolo,
    print_dataset_challenges
)

DATASET_ROOT = '/path/to/VisDrone'  # <-- update this
OUTPUT_DIR   = '../outputs'
os.makedirs(OUTPUT_DIR + '/visualizations', exist_ok=True)
print('Setup complete.')

## Task 01 – Dataset Understanding


In [ ]:
# Print known challenges
print_dataset_challenges()

# Analyse train split
stats = analyze_dataset(DATASET_ROOT, 'train')
print(f"\nImages : {stats['n_images']:,}")
print(f"Persons: {stats['target_class_counts']['person']:,}")
print(f"Cars   : {stats['target_class_counts']['car']:,}")
print(f"Avg obj/img: {np.mean(stats['objects_per_image']):.1f}")

In [ ]:
# Plot statistics
plot_dataset_stats(stats, save_dir=OUTPUT_DIR + '/visualizations')

In [ ]:
# Visualise 3 sample images with bounding boxes
img_dir = Path(DATASET_ROOT) / 'images' / 'train'
ann_dir = Path(DATASET_ROOT) / 'annotations' / 'train'

for i, ip in enumerate(sorted(img_dir.glob('*.jpg'))[:3]):
    ap = ann_dir / (ip.stem + '.txt')
    if ap.exists():
        visualize_sample(str(ip), str(ap),
                         save_path=f'{OUTPUT_DIR}/visualizations/sample_{i+1}.png')

In [ ]:
# Convert to YOLO format (run once)
YOLO_DATA = DATASET_ROOT + '/yolo_format'
convert_to_yolo(DATASET_ROOT, YOLO_DATA, splits=['train', 'val'])
print('Conversion complete.')

## Task 02 – Model Training


In [ ]:
from src.train import build_data_yaml, train, validate

data_yaml = build_data_yaml(YOLO_DATA)

# Train — adjust epochs/batch for your hardware
best_weights = train({
    'model':   'yolov8m.pt',
    'data':    data_yaml,
    'epochs':  50,     # increase for better results
    'imgsz':   640,
    'batch':   16,     # reduce to 8 if OOM
    'device':  '',     # auto
    'project': OUTPUT_DIR + '/runs',
    'name':    'visdrone_yolov8m',
})
print('Best weights:', best_weights)

In [ ]:
# Validate
metrics = validate(best_weights, data_yaml)
print(metrics)

## Task 03 – Detection & Human Counting


In [ ]:
from src.detect import DroneDetector, Visualizer, run_on_folder

# Use best_weights from training or a pretrained model
WEIGHTS = best_weights  # or 'yolov8m.pt'

detector = DroneDetector(weights=WEIGHTS, conf=0.30, iou=0.45)

# Run on a single image
test_img = sorted((Path(DATASET_ROOT) / 'images' / 'val').glob('*.jpg'))[0]
img, result = detector.detect_file(str(test_img))

print(f'Persons: {result.person_count}')
print(f'Cars   : {result.car_count}')
print(f'FPS    : {1000/result.inference_ms:.1f}')

annotated = Visualizer.draw(img, result)
plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title(f'Persons: {result.person_count}  |  Cars: {result.car_count}')
plt.show()

In [ ]:
# Batch inference on val set
val_dir = Path(DATASET_ROOT) / 'images' / 'val'
all_results = run_on_folder(detector, str(val_dir),
                             OUTPUT_DIR + '/detections')

total_persons = sum(r.person_count for r in all_results)
total_cars    = sum(r.car_count    for r in all_results)
print(f'Total persons across val set: {total_persons:,}')
print(f'Total cars   across val set: {total_cars:,}')

## Task 04 – Object Tracking (Bonus)


In [ ]:
from src.tracking import ByteTrackWrapper, SORTTracker, compute_tracking_stats
from src.detect import run_on_video

tracker = ByteTrackWrapper(max_age=30, min_hits=3)

# Run on a video (update path as needed)
VIDEO_PATH = '/path/to/drone_video.mp4'
out_vid    = OUTPUT_DIR + '/tracking/tracked_output.mp4'
os.makedirs(OUTPUT_DIR + '/tracking', exist_ok=True)

if Path(VIDEO_PATH).exists():
    track_results = run_on_video(detector, VIDEO_PATH, out_vid, tracker=tracker)
    tracking_stats = compute_tracking_stats(track_results)
    print(tracking_stats)
else:
    print('No video found. Skipping tracking demo.')

## Task 05 – Evaluation & Visualization


In [ ]:
from src.evaluate import (
    evaluate_predictions, plot_pr_curves,
    plot_count_timeline, plot_evaluation_dashboard,
    generate_summary_report
)
from src.dataset import parse_visdrone_annotation, TARGET_CLASS_MAP

# Count timeline
plot_count_timeline(all_results,
                    OUTPUT_DIR + '/visualizations/count_timeline.png')

# Build GT and pred annotation lists
gt_anns, pred_anns = [], []
val_imgs = sorted((Path(DATASET_ROOT) / 'images' / 'val').glob('*.jpg'))
val_anns = Path(DATASET_ROOT) / 'annotations' / 'val'

for fid, (ip, res) in enumerate(zip(val_imgs, all_results)):
    ap = val_anns / (ip.stem + '.txt')
    if not ap.exists(): continue
    img = cv2.imread(str(ip))
    for obj in parse_visdrone_annotation(str(ap)):
        if obj['category'] not in TARGET_CLASS_MAP: continue
        x, y, bw, bh = obj['bbox']
        gt_anns.append({'image_id': fid,
                         'class_id': TARGET_CLASS_MAP[obj['category']],
                         'bbox': [x, y, x+bw, y+bh]})
    for det in res.detections:
        pred_anns.append({'image_id': fid, 'class_id': det.class_id,
                           'bbox': det.bbox, 'score': det.confidence})

eval_results = evaluate_predictions(gt_anns, pred_anns, iou_threshold=0.5)
plot_pr_curves(eval_results, OUTPUT_DIR + '/visualizations/pr_curves.png')
plot_evaluation_dashboard(eval_results, all_results,
                           OUTPUT_DIR + '/visualizations/dashboard.png')

report = generate_summary_report(eval_results,
                                   save_path=OUTPUT_DIR + '/evaluation_report.json')

## Strengths, Limitations & Challenges

### Strengths
- YOLOv8 offers real-time inference suitable for drone applications
- Per-class independent tracking allows robust crowd counting
- Mosaic + copy-paste augmentation helps with small-object detection
- ByteTrack / SORT handles moderate occlusion with Kalman prediction

### Limitations
- Objects smaller than ~10 px remain difficult even with tiling
- Model struggles under heavy crowd occlusion
- Fixed confidence threshold may miss low-visibility objects
- CPU-only inference is too slow for real-time video

### Challenges Faced
- VisDrone class imbalance (pedestrians >> other classes)
- Tiny bounding boxes require carefully tuned NMS and augmentation
- Aerial perspective causes unusual object aspect ratios
- Converting VisDrone annotations to YOLO format required custom filtering logic
